<a href="https://colab.research.google.com/github/DorothyOduor/Markov-Chain-Model/blob/main/MarkovChainModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd

# The default path to your My Drive folder is always /content/drive/MyDrive/
file_path = '/content/drive/MyDrive/Colab Notebooks/JUNE-JULY.csv'

# Load the dataset
df = pd.read_csv(file_path)
df.head()


,month_name,user_pseudo_id,ga_session_id,ga_session_number,page_view_count,event_count,engagement_time_msec,session_duration,unique_pages,path_length,...,operating_system,country,region,city,day_of_week,hour_of_day,is_weekend,markov_path_sequence,application_submitted,drop_off
0,July,1.000118e+09,1783439710,1,1,5,30109,36,1,1,...,iOS,Kenya,Nairobi County,Nairobi,3,15,0,/ > / > / > / > /,0,1
1,July,1.000196e+09,1783700234,1,1,4,38,0,1,1,...,Android,Kenya,Nairobi County,Nairobi,6,16,0,/applications/ > /applications/ > /application...,0,1
2,July,1.000196e+09,1783929714,2,1,2,0,0,1,1,...,Android,Kenya,Nairobi County,Nairobi,2,8,0,/ > /,0,1
3,July,1.000256e+08,1784616575,1,3,6,2054,706,1,3,...,iOS,Kenya,Nairobi County,Nairobi,3,7,0,/ > / > / > / > / > /,0,1
4,July,1.000256e+08,1785177362,2,1,2,0,0,1,1,...,iOS,Kenya,Nairobi County,Nairobi,2,18,0,/ > /,0,1


In [4]:
print(f"Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Initial Shape: 34325 rows, 32 columns


**DATA** **CLEANING**

i. Removing Invalid and Empty Sessions

In [5]:
# Drop sessions missing essential identifiers or navigation sequences
df = df.dropna(subset=['user_pseudo_id', 'ga_session_id', 'markov_path_sequence'])

In [6]:
# Remove sessions where path_length is 0 or session_duration is negative
df = df[(df['path_length'] > 0) & (df['session_duration'] >= 0)]

ii. Handle Missing & Null Values

In [7]:
# Fill categorical fields with explicit default labels
categorical_cols = {
    'traffic_source': '(direct)',
    'traffic_medium': '(none)',
    'default_channel_group': 'Direct',
    'device_category': 'unknown',
    'browser': 'unknown',
    'operating_system': 'unknown',
    'country': 'Unknown',
    'region': 'Unknown',
    'city': 'Unknown',
    'landing_page': '/unknown',
    'exit_page': '/unknown'
}
df = df.fillna(value=categorical_cols)

In [8]:
# Ensure numeric metrics have no NaNs
numeric_cols = ['engagement_time_msec', 'session_duration', 'unique_pages', 'path_length']
df[numeric_cols] = df[numeric_cols].fillna(0)

iii. Handle Outliers (Bot & Crawler Filtering)

In [10]:
# Web analytics data often contains bots (e.g., sessions with 500+ page views or days of duration)
# We filter out extreme upper percentiles (e.g., top 0.1% or hard caps)
max_duration_seconds = 86400  # 24 hours cap
max_page_views = 200         # Reasonable cap for high-intent manual browsing

df = df[(df['session_duration'] <= max_duration_seconds) & (df['path_length'] <= max_page_views)]

iv. Clean & Standardize Markov Path Strings

In [11]:
def clean_path_sequence(sequence):
    if not isinstance(sequence, str):
        return ""
    # Split paths, strip whitespace, remove empty elements
    paths = [p.strip() for p in sequence.split('>') if p.strip()]

In [15]:
def clean_path_sequence(sequence):
    if not isinstance(sequence, str):
        return ""
    # Split paths, strip whitespace, remove empty elements
    paths = [p.strip() for p in sequence.split('>') if p.strip()]
    # Remove consecutive duplicate page views (e.g., Home > Home > Admissions -> Home > Admissions)
    deduped = [paths[i] for i in range(len(paths)) if i == 0 or paths[i] != paths[i-1]]

    return " > ".join(deduped)

df['markov_path_sequence_clean'] = df['markov_path_sequence'].apply(clean_path_sequence)

# Remove rows where sequence became empty after cleaning
df = df[df['markov_path_sequence_clean'] != ""]

v. Recalculate Sequence Lengths & Verification

In [16]:
df['clean_step_count'] = df['markov_path_sequence_clean'].apply(lambda x: len(x.split(' > ')))

print(f"Cleaned Shape: {df.shape[0]} rows remaining ({df.shape[0]/45399:.1%} of raw data retained)")
df.head()

Cleaned Shape: 32380 rows remaining (71.3% of raw data retained)


,month_name,user_pseudo_id,ga_session_id,ga_session_number,page_view_count,event_count,engagement_time_msec,session_duration,unique_pages,path_length,...,region,city,day_of_week,hour_of_day,is_weekend,markov_path_sequence,application_submitted,drop_off,markov_path_sequence_clean,clean_step_count
0,July,1.000118e+09,1783439710,1,1,5,30109,36,1,1,...,Nairobi County,Nairobi,3,15,0,/ > / > / > / > /,0,1,/,1
1,July,1.000196e+09,1783700234,1,1,4,38,0,1,1,...,Nairobi County,Nairobi,6,16,0,/applications/ > /applications/ > /application...,0,1,/applications/,1
2,July,1.000196e+09,1783929714,2,1,2,0,0,1,1,...,Nairobi County,Nairobi,2,8,0,/ > /,0,1,/,1
3,July,1.000256e+08,1784616575,1,3,6,2054,706,1,3,...,Nairobi County,Nairobi,3,7,0,/ > / > / > / > / > /,0,1,/,1
4,July,1.000256e+08,1785177362,2,1,2,0,0,1,1,...,Nairobi County,Nairobi,2,18,0,/ > /,0,1,/,1
